# Lezione 5 · Le CNN, l'arte dello sguardo

La lente che scorre sulla carta, i conti dei pesi e delle moltiplicazioni, il max pooling.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

## La lente su una riga di pixel

Il filtro (−1; 0; 1) toglie il pixel a sinistra e aggiunge quello a destra.

In [2]:
filtro = np.array([-1, 0, 1])
for riga, atteso in [((3, 3, 3), 0), ((3, 3, 9), 6), ((3, 3, 4), 1), ((9, 3, 3), -6)]:
    controlla(f"riga {riga}", np.dot(filtro, riga), atteso)

✓  riga (3, 3, 3): calcolato 0, nel libro 0
✓  riga (3, 3, 9): calcolato 6, nel libro 6
✓  riga (3, 3, 4): calcolato 1, nel libro 1
✓  riga (9, 3, 3): calcolato -6, nel libro -6


## La lente che scorre su un'immagine

Una piccola carta 6×6 con una crocetta, e un filtro 3×3 che cerca proprio quella forma.

In [3]:
carta = np.zeros((6, 6))
carta[2, 3] = carta[3, 2] = carta[3, 3] = carta[3, 4] = carta[4, 3] = 1   # una crocetta
croce = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])

mappa = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        mappa[i, j] = np.sum(carta[i:i+3, j:j+3] * croce)   # moltiplica e somma, sempre con gli stessi pesi
print(mappa)
print("la lente si accende di più in", np.unravel_index(mappa.argmax(), mappa.shape))

[[0. 0. 1. 0.]
 [0. 2. 2. 2.]
 [1. 2. 5. 2.]
 [0. 2. 2. 2.]]
la lente si accende di più in (np.int64(2), np.int64(2))


## Pesi e conti

In [4]:
pesi_lenti = 64 * 27
moltiplicazioni_lenti = 64 * 27 * 224 * 224
pesi_denso = 64 * 224 * 224 * 3
print(f"64 lenti: {pesi_lenti} pesi, {moltiplicazioni_lenti/1e6:.0f} milioni di moltiplicazioni")
print(f"64 neuroni densi: {pesi_denso/1e6:.1f} milioni di pesi e altrettante moltiplicazioni")
controlla("quante volte più conti fanno le lenti", moltiplicazioni_lenti / pesi_denso, 9, 0.1)
controlla("passi di AlexNet (migliaia)", 1_200_000 * 90 / 128 / 1e3, 844, 1)

64 lenti: 1728 pesi, 87 milioni di moltiplicazioni
64 neuroni densi: 9.6 milioni di pesi e altrettante moltiplicazioni
✓  quante volte più conti fanno le lenti: calcolato 9, nel libro 9
✓  passi di AlexNet (migliaia): calcolato 843.8, nel libro 844


## Kata per casa

**Esercizio 1.** I parametri di una lente 3×3 su 3 canali, di 64 lenti, e di 64 neuroni densi.

In [5]:
controlla("parametri di un filtro", 3 * 3 * 3 + 1, 28, 0)
controlla("parametri di 64 filtri", 64 * 28, 1792, 0)
controlla("pesi di 64 neuroni densi", 224 * 224 * 3 * 64, 9_633_792, 0)

✓  parametri di un filtro: calcolato 28, nel libro 28
✓  parametri di 64 filtri: calcolato 1792, nel libro 1792
✓  pesi di 64 neuroni densi: calcolato 9.634e+06, nel libro 9.63379e+06


**Esercizio 2.** Il max pooling 2×2 della mappa 4×4.

In [6]:
mappa = np.array([[1, 0, 2, 3], [4, 6, 6, 8], [3, 1, 1, 0], [1, 2, 2, 4]])
pool = mappa.reshape(2, 2, 2, 2).max(axis=(1, 3))
print(pool)
assert pool.tolist() == [[6, 8], [3, 4]]

[[6 8]
 [3 4]]
